<a href="https://colab.research.google.com/github/Deva2013/airline-disruption-management-system/blob/main/Airline_Disruption_Phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Environment Setup ──────────────────────────────────────────────────────
import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

# Install Hugging Face Hub client
!pip install -q huggingface_hub

from huggingface_hub import HfApi, login

# ── Authenticate with Hugging Face using the Colab secret ──────────────────
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

HF_REPO_ID = "Dev123Hug456Face/airline-disruption-data"
hf_api = HfApi()

# ── Directory structure ──────────────────────────────────────────────────
# Everything lives on LOCAL Colab disk (/content) — fast, reliable, no
# Drive quota issues. This does NOT persist across sessions; that's
# expected. Finished "gold" outputs get pushed to Hugging Face Hub at the
# end of each stage instead of being written to Drive.

BASE_DIR       = Path('/content/airline-disruption')
DIR_RAW        = BASE_DIR / 'data' / 'raw'
DIR_PROCESSED  = BASE_DIR / 'data' / 'processed'
DIR_WEATHER    = BASE_DIR / 'data' / 'weather'

for d in [DIR_RAW, DIR_PROCESSED, DIR_WEATHER]:
    d.mkdir(parents=True, exist_ok=True)

print('Environment ready.')
print(f'Base directory: {BASE_DIR}')
print(f'HF repo target: {HF_REPO_ID}')

Environment ready.
Base directory: /content/airline-disruption
HF repo target: Dev123Hug456Face/airline-disruption-data


In [ ]:
# ── Pull the verified BTS dataset from Hugging Face Hub ────────────────────
from huggingface_hub import hf_hub_download

bts_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="bts_cleaned.parquet",
    repo_type="dataset",
    local_dir=DIR_PROCESSED,
)

print(f"Downloaded to: {bts_path}")

# Quick verification
df = pd.read_parquet(bts_path)
print(f"\nRows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
print(f"\nColumn list: {df.columns.tolist()}")

bts_cleaned.parquet: reconstructing file:   0%|          |  0.00B /  355MB            

bts_cleaned.parquet: downloading bytes:           |  0.00B            

Downloaded to: /content/airline-disruption/data/processed/bts_cleaned.parquet

Rows: 10,504,936
Columns: 41
Year range: 2022 - 2024

Column list: ['Year', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Airline', 'FlightNumber', 'OperatingAirline', 'OperatingAirlineCode', 'Origin', 'OriginCityName', 'OriginState', 'Dest', 'DestCityName', 'DestState', 'CRSDepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'TaxiOut', 'TaxiIn', 'CRSArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'Cancelled', 'CancellationCode', 'Diverted', 'AirTime', 'Distance', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'CancellationReason', 'PrimaryDelayCause', 'SeverityTier', 'ScheduledDepHour', 'Season', 'IsWeekend']


In [ ]:
# ── Dataset overview ────────────────────────────────────────────────────
total     = len(df)
cancelled = df['Cancelled'].sum()
dep_del   = df['DepDel15'].sum()
arr_del   = df['ArrDel15'].sum()

print('=' * 50)
print('  DATASET OVERVIEW')
print('=' * 50)
print(f'  Flights       : {total:,}')
print(f'  Date range    : {df["FlightDate"].min()} → {df["FlightDate"].max()}')
print(f'  Airlines      : {df["Airline"].nunique()}')
print(f'  Cancellation  : {cancelled/total*100:.2f}%  ({cancelled:,})')
print(f'  Dep delay≥15m : {dep_del/total*100:.2f}%  ({dep_del:,})')
print(f'  Arr delay≥15m : {arr_del/total*100:.2f}%  ({arr_del:,})')
print()
print('Severity breakdown:')
print(df['SeverityTier'].value_counts().to_string())

  DATASET OVERVIEW
  Flights       : 10,504,936
  Date range    : 2022-01-01 00:00:00 → 2024-12-31 00:00:00
  Airlines      : 10
  Cancellation  : 1.81%  (190,145)
  Dep delay≥15m : 20.04%  (2,105,664)
  Arr delay≥15m : 20.45%  (2,148,534)

Severity breakdown:
SeverityTier
On Time        8166257
Minor          1140970
Significant     697812
Severe          309752
Cancelled       190145


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Fetch Historical METAR Weather Data (IEM ASOS Archive)
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Pull hourly weather observations for all 10 hub airports,
#          2022-2024, from the Iowa Environmental Mesonet (IEM) ASOS
#          archive — a true historical data source (unlike
#          aviationweather.gov, which only serves the last 15 days).
#
# Storage: Raw per-airport-year CSVs are cached on LOCAL Colab disk
#          (BASE_DIR/data/metar_raw), NOT Google Drive. This data is
#          disposable/re-fetchable, so it doesn't need to persist
#          beyond this session.
# ═══════════════════════════════════════════════════════════════════════

import io

IEM_URL = 'https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py'
METAR_RAW = BASE_DIR / 'data' / 'metar_raw'
METAR_RAW.mkdir(exist_ok=True)

# Same 10 hub airports used in the BTS dataset, mapped to ICAO codes
# (ICAO codes are the 4-letter identifiers IEM's station data references)
AIRPORT_ICAO = {
    'JFK': 'KJFK', 'ORD': 'KORD', 'ATL': 'KATL', 'LAX': 'KLAX', 'DFW': 'KDFW',
    'SFO': 'KSFO', 'EWR': 'KEWR', 'MIA': 'KMIA', 'SEA': 'KSEA', 'BOS': 'KBOS',
}
START_YEAR, END_YEAR = 2022, 2024


def iem_station(icao):
    """IEM station IDs drop the leading 'K' for CONUS airports (KJFK -> JFK)."""
    return icao[1:] if icao.startswith('K') and len(icao) == 4 else icao


def fetch_iem_year(icao, year, max_retries=3):
    """
    Fetch one full year of METAR data for one airport from IEM.
    Retries with backoff on failure, and validates the response is
    real CSV data (not an HTML error/rate-limit page) before returning it.
    """
    params = [
        ('station', iem_station(icao)),
        ('data', 'all'),
        ('tz', 'Etc/UTC'),
        ('format', 'comma'),
        ('latlon', 'no'),
        ('sts', f'{year}-01-01T00:00:00Z'),
        ('ets', f'{year+1}-01-01T00:00:00Z' if year < END_YEAR else f'{year}-12-31T23:59:59Z'),
    ]
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(
                IEM_URL, params=params,
                headers={'User-Agent': 'airline-disruption-research (ASU student project)'},
                timeout=90
            )
            r.raise_for_status()
            text = r.text

            # Validation: IEM prefixes real responses with a few "#DEBUG:"
            # comment lines before the actual CSV header. Check the header
            # shows up near the top rather than requiring it be line 1 —
            # this rejects bad/empty responses instead of crashing later.
            if 'station,valid' in text[:600]:
                return text
            else:
                print(f'    [warn] {icao} {year}: unexpected response '
                      f'(attempt {attempt}), first 80 chars: {text[:80]!r}')
        except Exception as e:
            print(f'    [warn] {icao} {year}: {e} (attempt {attempt})')
        time.sleep(3 * attempt)  # back off longer with each retry
    return None


# ── Main fetch loop: one request per airport per year (30 total) ──────────
print('Fetching METAR data from IEM ASOS archive (local disk cache)...')
all_frames = []
failed = []

for iata, icao in AIRPORT_ICAO.items():
    print(f'  {iata} ({icao})')
    for year in range(START_YEAR, END_YEAR + 1):
        cache_file = METAR_RAW / f'{icao}_{year}.csv'

        # Reuse cached file if it exists and looks valid; otherwise fetch fresh
        if cache_file.exists():
            text = cache_file.read_text()
            if 'station,valid' not in text[:600]:
                cache_file.unlink()  # discard invalid cached file
                text = None
        else:
            text = None

        if text is None:
            text = fetch_iem_year(icao, year)
            if text:
                cache_file.write_text(text)
            time.sleep(1.5)  # be polite to IEM's server between requests

        if not text:
            failed.append((icao, year))
            continue

        # comment='#' skips IEM's "#DEBUG:" preamble lines automatically
        df_year = pd.read_csv(io.StringIO(text), na_values=['M'], comment='#')
        df_year['IATA'] = iata
        all_frames.append(df_year)

# ── Combine all airport-years into one dataframe ───────────────────────────
wx_raw = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
print(f'\nTotal METAR records: {len(wx_raw):,}')
if failed:
    print(f'⚠️  Failed after retries: {failed}')
wx_raw.head(2)

Fetching METAR data from IEM ASOS archive (local disk cache)...
  JFK (KJFK)
  ORD (KORD)
    [warn] KORD 2023: 503 Server Error: Service Unavailable for url: https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station=ORD&data=all&tz=Etc%2FUTC&format=comma&latlon=no&sts=2023-01-01T00%3A00%3A00Z&ets=2024-01-01T00%3A00%3A00Z (attempt 1)
    [warn] KORD 2023: 503 Server Error: Service Unavailable for url: https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station=ORD&data=all&tz=Etc%2FUTC&format=comma&latlon=no&sts=2023-01-01T00%3A00%3A00Z&ets=2024-01-01T00%3A00%3A00Z (attempt 2)
  ATL (KATL)
  LAX (KLAX)


/tmp/ipykernel_1177/1516899102.py:105: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_year = pd.read_csv(io.StringIO(text), na_values=['M'], comment='#')


  DFW (KDFW)
  SFO (KSFO)
  EWR (KEWR)
  MIA (KMIA)


/tmp/ipykernel_1177/1516899102.py:105: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_year = pd.read_csv(io.StringIO(text), na_values=['M'], comment='#')


  SEA (KSEA)
  BOS (KBOS)

Total METAR records: 3,381,319


,station,valid,tmpf,dwpf,relh,drct,sknt,p01i,alti,mslp,...,ice_accretion_1hr,ice_accretion_3hr,ice_accretion_6hr,peak_wind_gust,peak_wind_drct,peak_wind_time,feel,metar,snowdepth,IATA
0,JFK,2022-01-01 00:00,49.0,48.0,96.32,200.0,6.0,NaN,NaN,1014.9,...,NaN,NaN,NaN,NaN,NaN,NaN,46.0,METAR JFK 010000Z AUTO 20006KT BR 09/09 RMK AO...,NaN,JFK
1,JFK,2022-01-01 00:00,NaN,NaN,NaN,200.0,5.0,NaN,29.97,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KJFK 010000Z AUTO 20005KT 8SM BKN004 OVC014 09...,NaN,JFK


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Clean and Aggregate METAR Data to Hourly Airport-Level Records
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Convert raw ~5-minute-interval METAR observations into one
#          clean row per (airport, date, hour) — matching the grain
#          needed to join against the BTS flight data later.
#
# Key logic: IEM reports every ~5 minutes, so each airport-hour has
#            multiple raw observations. We aggregate to hourly by
#            keeping the WORST (most disruptive) condition seen in
#            that hour — e.g. minimum visibility, maximum wind —
#            since that's what actually matters for flight delays.
# ═══════════════════════════════════════════════════════════════════════

def clean_metars(df):
    df = df.copy()

    # Parse observation timestamp; drop rows where it's unparseable
    df['ObsTime'] = pd.to_datetime(df['valid'], errors='coerce', utc=True)
    df = df.dropna(subset=['ObsTime'])

    # Convert/rename raw IEM fields to clean, analysis-ready columns
    df['TempC']       = (pd.to_numeric(df['tmpf'], errors='coerce') - 32) * 5 / 9
    df['WindSpeedKt'] = pd.to_numeric(df['sknt'], errors='coerce')
    df['WindGustKt']  = pd.to_numeric(df['gust'], errors='coerce')
    df['VisibSM']     = pd.to_numeric(df['vsby'], errors='coerce')

    # Disruption indicator flags (thresholds based on common aviation
    # operational impact levels)
    df['IsLowVis']   = (df['VisibSM']    <  3).astype(int)
    df['IsHighWind'] = (df['WindSpeedKt']>= 25).astype(int)
    df['IsGust']     = (df['WindGustKt'] >= 35).astype(int)

    # Extract date and hour for joining against flight-level BTS data
    df['Date']    = df['ObsTime'].dt.date
    df['DepHour'] = df['ObsTime'].dt.hour

    # ── Aggregate multiple observations per airport-hour into one row ──
    # Uses the WORST condition seen in the hour (min visibility, max
    # wind/gust) since that's most relevant to whether flights were
    # actually disrupted during that hour.
    agg = df.groupby(['IATA', 'Date', 'DepHour']).agg(
        TempC=('TempC', 'mean'),
        WindSpeedKt=('WindSpeedKt', 'max'),
        WindGustKt=('WindGustKt', 'max'),
        VisibSM=('VisibSM', 'min'),
        IsLowVis=('IsLowVis', 'max'),
        IsHighWind=('IsHighWind', 'max'),
        IsGust=('IsGust', 'max'),
    ).reset_index()

    # Classify flight-rules category based on worst visibility in the hour
    def flight_cat(vis):
        if pd.isna(vis): return 'Unknown'
        if vis < 1:  return 'LIFR'   # Low Instrument Flight Rules
        if vis < 3:  return 'IFR'    # Instrument Flight Rules
        if vis < 5:  return 'MVFR'   # Marginal Visual Flight Rules
        return 'VFR'                 # Visual Flight Rules (normal)
    agg['FlightCategory'] = agg['VisibSM'].apply(flight_cat)
    agg['IsFogOrIFR'] = agg['FlightCategory'].isin(['IFR', 'LIFR']).astype(int)

    return agg


# ── Run cleaning and save locally ──────────────────────────────────────
wx = clean_metars(wx_raw)

wx_path = DIR_WEATHER / 'metar_clean.parquet'
wx.to_parquet(wx_path, index=False)

print(f'Clean METAR records (one row per airport-hour): {len(wx):,}')
print(f'Saved locally to: {wx_path}')
print()
print('Flight category breakdown:')
print(wx['FlightCategory'].value_counts().to_string())

Clean METAR records (one row per airport-hour): 263,034
Saved locally to: /content/airline-disruption/data/weather/metar_clean.parquet

Flight category breakdown:
FlightCategory
VFR     243922
MVFR      7746
IFR       7493
LIFR      3873


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Upload Clean METAR Dataset to Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Persist the finished, cleaned weather dataset permanently.
#          This is a "gold" file (small, finished, analysis-ready) —
#          unlike the raw METAR CSVs, which stay local/disposable.
# ═══════════════════════════════════════════════════════════════════════

hf_api.upload_file(
    path_or_fileobj=str(wx_path),
    path_in_repo="metar_clean.parquet",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)

print("✅ metar_clean.parquet uploaded to Hugging Face Hub")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ather/metar_clean.parquet:  72%|#######1  |  536kB /  745kB            

✅ metar_clean.parquet uploaded to Hugging Face Hub


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Join BTS Flight Data with METAR Weather Data
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Attach weather conditions to each flight, based on the
#          ORIGIN airport and the flight's SCHEDULED departure hour.
#          This is what lets the disruption engine later distinguish
#          weather-driven delays from other causes.
#
# Join keys: Origin airport (IATA) + FlightDate + ScheduledDepHour
#            match against wx's IATA + Date + DepHour.
# ═══════════════════════════════════════════════════════════════════════

# ── Prepare join keys on both sides ────────────────────────────────────
# BTS side: use the flight's origin airport, date, and scheduled dep hour
df_join = df.copy()
df_join['JoinDate'] = df_join['FlightDate'].dt.date  # match wx's date type

# Weather side: already has IATA, Date, DepHour from the cleaning step
wx_join = wx.copy()

# ── Perform the join ────────────────────────────────────────────────────
# Left join: keep ALL flights, even if weather data is missing for that
# airport-hour (rare given ~100% coverage, but don't want to silently
# drop flights if it happens)
bts_weather = df_join.merge(
    wx_join,
    left_on=['Origin', 'JoinDate', 'ScheduledDepHour'],
    right_on=['IATA', 'Date', 'DepHour'],
    how='left'
)

# Drop the now-redundant join key columns from the weather side
bts_weather = bts_weather.drop(columns=['JoinDate', 'IATA', 'Date', 'DepHour'])

# ── Check join quality ──────────────────────────────────────────────────
matched = bts_weather['FlightCategory'].notna().sum()
total = len(bts_weather)

print(f'Total flights: {total:,}')
print(f'Matched with weather data: {matched:,} ({matched/total*100:.2f}%)')
print(f'Unmatched (no weather record for that airport-hour): {total - matched:,}')
print()
print('Sample of joined data:')
bts_weather[['FlightDate', 'Origin', 'ScheduledDepHour', 'TempC',
             'WindSpeedKt', 'VisibSM', 'FlightCategory']].head()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Environment Setup
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Set up local working directories and authenticate with
#          Hugging Face Hub. No Google Drive is used anywhere in this
#          pipeline — all working data lives on local Colab disk
#          (/content), and finished outputs are pushed to Hugging Face
#          Hub for permanent storage.
# ═══════════════════════════════════════════════════════════════════════

import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

# Install Hugging Face Hub client
!pip install -q huggingface_hub

from huggingface_hub import HfApi, login, hf_hub_download

# ── Authenticate with Hugging Face using the Colab secret ──────────────────
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

HF_REPO_ID = "Dev123Hug456Face/airline-disruption-data"
hf_api = HfApi()

# ── Directory structure (local Colab disk only) ─────────────────────────
BASE_DIR       = Path('/content/airline-disruption')
DIR_RAW        = BASE_DIR / 'data' / 'raw'
DIR_PROCESSED  = BASE_DIR / 'data' / 'processed'
DIR_WEATHER    = BASE_DIR / 'data' / 'weather'

for d in [DIR_RAW, DIR_PROCESSED, DIR_WEATHER]:
    d.mkdir(parents=True, exist_ok=True)

print('Environment ready.')
print(f'Base directory: {BASE_DIR}')
print(f'HF repo target: {HF_REPO_ID}')

Environment ready.
Base directory: /content/airline-disruption
HF repo target: Dev123Hug456Face/airline-disruption-data


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Pull BTS Flight Data from Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Restore the cleaned BTS dataset (previously uploaded) into
#          this fresh session, without needing to reprocess the 36
#          raw ZIP files again.
# ═══════════════════════════════════════════════════════════════════════

bts_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="bts_cleaned.parquet",
    repo_type="dataset",
    local_dir=DIR_PROCESSED,
)

df = pd.read_parquet(bts_path)

print(f"Downloaded to: {bts_path}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Downloaded to: /content/airline-disruption/data/processed/bts_cleaned.parquet
Rows: 10,504,936
Columns: 41


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Pull Clean METAR Weather Data from Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Restore the cleaned, hourly-aggregated weather dataset
#          (previously uploaded) into this fresh session, without
#          needing to re-fetch from IEM's ASOS archive again.
# ═══════════════════════════════════════════════════════════════════════

wx_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="metar_clean.parquet",
    repo_type="dataset",
    local_dir=DIR_WEATHER,
)

wx = pd.read_parquet(wx_path)

print(f"Downloaded to: {wx_path}")
print(f"Rows: {len(wx):,}")
print(f"Columns: {wx.shape[1]}")

Downloaded to: /content/airline-disruption/data/weather/metar_clean.parquet
Rows: 263,034
Columns: 12


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Reduce Memory Footprint Before Joining (prevents OOM crash)
# ═══════════════════════════════════════════════════════════════════════
# Purpose: The previous join attempt crashed from running out of RAM.
#          Root cause: several BTS columns are stored as generic 'object'
#          (string) dtype, which is memory-heavy in pandas — especially
#          across 10.5M rows. Converting repetitive text columns to
#          'category' dtype stores each unique value once instead of
#          once per row, cutting memory use dramatically before we
#          attempt the merge.
# ═══════════════════════════════════════════════════════════════════════

import gc

# Columns with a small number of repeated values — ideal for 'category'
category_cols = [
    'Airline', 'OperatingAirline', 'OperatingAirlineCode',
    'Origin', 'OriginCityName', 'OriginState',
    'Dest', 'DestCityName', 'DestState',
    'CancellationReason', 'PrimaryDelayCause', 'SeverityTier', 'Season',
]

for col in category_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Same treatment for the weather side
if 'FlightCategory' in wx.columns:
    wx['IATA'] = wx['IATA'].astype('category')
    wx['FlightCategory'] = wx['FlightCategory'].astype('category')

# Force garbage collection to release any freed memory immediately
gc.collect()

print('Memory optimization complete.')
print(f'df memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print(f'wx memory usage: {wx.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Memory optimization complete.
df memory usage: 1.51 GB
wx memory usage: 0.03 GB


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Check Available Memory, Then Join BTS + Weather Data
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Confirm there's enough RAM headroom before attempting the
#          merge (which crashed the session last time before memory
#          optimization). Then join BTS flights with weather data,
#          matching each flight's ORIGIN airport + FlightDate +
#          ScheduledDepHour against the weather record for that
#          airport-hour.
# ═══════════════════════════════════════════════════════════════════════

# Check available RAM first
!free -h

print()
print('Proceeding with join...')

# ── Prepare join keys ────────────────────────────────────────────────────
df['JoinDate'] = df['FlightDate'].dt.date  # match wx's date type

# ── Perform the join (left join: keep ALL flights) ─────────────────────
bts_weather = df.merge(
    wx,
    left_on=['Origin', 'JoinDate', 'ScheduledDepHour'],
    right_on=['IATA', 'Date', 'DepHour'],
    how='left'
)

# Drop redundant join key columns from the weather side
bts_weather = bts_weather.drop(columns=['JoinDate', 'IATA', 'Date', 'DepHour'])

# Free up memory from intermediate objects no longer needed
gc.collect()

# ── Check join quality ──────────────────────────────────────────────────
matched = bts_weather['FlightCategory'].notna().sum()
total = len(bts_weather)

print(f'\nTotal flights: {total:,}')
print(f'Matched with weather data: {matched:,} ({matched/total*100:.2f}%)')
print(f'Unmatched: {total - matched:,}')
print(f'\nResult memory usage: {bts_weather.memory_usage(deep=True).sum() / 1e9:.2f} GB')

               total        used        free      shared  buff/cache   available
Mem:            12Gi       8.1Gi       2.7Gi       2.0Mi       1.8Gi       4.3Gi
Swap:             0B          0B          0B

Proceeding with join...

Total flights: 10,504,936
Matched with weather data: 5,896,424 (56.13%)
Unmatched: 4,608,512

Result memory usage: 2.72 GB


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Filter to Origin-Hub Flights, Then Re-Verify Weather Join
# ═══════════════════════════════════════════════════════════════════════
# Purpose: The previous join only matched 56% of flights, because the
#          BTS dataset includes flights arriving AT a hub from
#          non-hub origins — and weather data only covers the 10 hub
#          airports. Restricting to flights DEPARTING FROM a hub
#          (matching the project's departure-disruption framing)
#          should give near-100% weather match coverage.
# ═══════════════════════════════════════════════════════════════════════

TARGET_AIRPORTS = ['JFK', 'ORD', 'ATL', 'LAX', 'DFW', 'SFO', 'EWR', 'MIA', 'SEA', 'BOS']

# Keep only flights departing FROM one of the 10 target hub airports
bts_weather_scoped = bts_weather[bts_weather['Origin'].isin(TARGET_AIRPORTS)].copy()

matched = bts_weather_scoped['FlightCategory'].notna().sum()
total = len(bts_weather_scoped)

print(f'Flights departing from target hubs: {total:,} (was {len(bts_weather):,} before filtering)')
print(f'Matched with weather data: {matched:,} ({matched/total*100:.2f}%)')
print(f'Unmatched: {total - matched:,}')

Flights departing from target hubs: 5,896,606 (was 10,504,936 before filtering)
Matched with weather data: 5,896,424 (100.00%)
Unmatched: 182


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Free Memory and Save the Joined Dataset Locally
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Drop the old unfiltered join result (no longer needed) to
#          free memory, then save the final scoped BTS+weather dataset
#          to local disk as the "gold" output for this stage.
# ═══════════════════════════════════════════════════════════════════════

# Free the old unfiltered join — no longer needed, was using ~2.7GB
del bts_weather
gc.collect()

# Rename for clarity going forward
bts_weather = bts_weather_scoped
del bts_weather_scoped
gc.collect()

# Save locally first
joined_path = DIR_PROCESSED / 'bts_weather_joined.parquet'
bts_weather.to_parquet(joined_path, index=False)

print(f'Saved: {joined_path}')
print(f'Rows: {len(bts_weather):,}')
print(f'Columns: {bts_weather.shape[1]}')
print(f'Memory usage: {bts_weather.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Saved: /content/airline-disruption/data/processed/bts_weather_joined.parquet
Rows: 5,896,606
Columns: 50
Memory usage: 1.57 GB


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Upload Joined BTS+Weather Dataset to Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Persist the finished BTS+weather join permanently — this is
#          the "gold" dataset the disruption detection engine and
#          predictive model will build on next.
# ═══════════════════════════════════════════════════════════════════════

hf_api.upload_file(
    path_or_fileobj=str(joined_path),
    path_in_repo="bts_weather_joined.parquet",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)

print("✅ bts_weather_joined.parquet uploaded to Hugging Face Hub")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ts_weather_joined.parquet:   1%|          |  560kB /  112MB            

✅ bts_weather_joined.parquet uploaded to Hugging Face Hub


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Disruption Detection Engine
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Classify each flight's disruption using both the BTS-reported
#          delay cause (CarrierDelay, WeatherDelay, NASDelay, etc.) and
#          the actual weather conditions at departure (from our METAR
#          join) to catch cases where weather contributed to a delay
#          even when BTS attributed it to a different primary cause
#          (a known reporting quirk — airlines often under-report
#          weather as the cause when it's a contributing factor).
# ═══════════════════════════════════════════════════════════════════════

# ── Weather-Contributed Flag ────────────────────────────────────────────
# True if degraded weather conditions were present at departure time,
# REGARDLESS of what BTS listed as the "official" primary cause.
bts_weather['WeatherContributed'] = (
    (bts_weather['IsLowVis'] == 1) |
    (bts_weather['IsHighWind'] == 1) |
    (bts_weather['IsGust'] == 1) |
    (bts_weather['IsFogOrIFR'] == 1)
).astype(int)

# ── Refined Disruption Classification ───────────────────────────────────
# Combines BTS's official PrimaryDelayCause with our own weather signal
# to produce a more complete disruption type label.
def classify_disruption(row):
    if row['Cancelled'] == 1:
        return 'Cancelled'
    if row['Diverted'] == 1:
        return 'Diverted'
    if row['ArrDelayMinutes'] < 15:
        return 'On Time'
    # Delayed flight — determine primary driver
    if row['WeatherContributed'] == 1:
        return 'Weather-Related Delay'
    if pd.notna(row['PrimaryDelayCause']):
        return f"{row['PrimaryDelayCause']} Delay"
    return 'Other Delay'

bts_weather['DisruptionType'] = bts_weather.apply(classify_disruption, axis=1)

# ── Cascade Risk Flag ────────────────────────────────────────────────────
# Flights with LateAircraftDelay > 0 indicate the delay is a downstream
# effect of a PREVIOUS flight's lateness (the incoming aircraft was late)
# — this is a useful signal for identifying cascading disruption chains.
bts_weather['IsCascadeRisk'] = (bts_weather['LateAircraftDelay'] > 0).astype(int)

# ── Summary ──────────────────────────────────────────────────────────────
print('Disruption Type breakdown:')
print(bts_weather['DisruptionType'].value_counts().to_string())
print()
print(f'Weather-contributed flights: {bts_weather["WeatherContributed"].sum():,} '
      f'({bts_weather["WeatherContributed"].mean()*100:.2f}%)')
print(f'Cascade-risk flights (late aircraft): {bts_weather["IsCascadeRisk"].sum():,} '
      f'({bts_weather["IsCascadeRisk"].mean()*100:.2f}%)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Environment Setup
# ═══════════════════════════════════════════════════════════════════════
import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

!pip install -q huggingface_hub

from huggingface_hub import HfApi, login, hf_hub_download
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

HF_REPO_ID = "Dev123Hug456Face/airline-disruption-data"
hf_api = HfApi()

BASE_DIR       = Path('/content/airline-disruption')
DIR_RAW        = BASE_DIR / 'data' / 'raw'
DIR_PROCESSED  = BASE_DIR / 'data' / 'processed'
DIR_WEATHER    = BASE_DIR / 'data' / 'weather'

for d in [DIR_RAW, DIR_PROCESSED, DIR_WEATHER]:
    d.mkdir(parents=True, exist_ok=True)

print('Environment ready.')

Environment ready.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Pull Joined BTS+Weather Dataset from Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Restore the already-joined, origin-hub-scoped dataset
#          (5.9M flights) directly — skips re-downloading BTS,
#          re-fetching weather, and re-running the join.
# ═══════════════════════════════════════════════════════════════════════

joined_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="bts_weather_joined.parquet",
    repo_type="dataset",
    local_dir=DIR_PROCESSED,
)

bts_weather = pd.read_parquet(joined_path)

print(f"Downloaded to: {joined_path}")
print(f"Rows: {len(bts_weather):,}")
print(f"Columns: {bts_weather.shape[1]}")
print(f"Memory usage: {bts_weather.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Downloaded to: /content/airline-disruption/data/processed/bts_weather_joined.parquet
Rows: 5,896,606
Columns: 50
Memory usage: 1.52 GB


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Disruption Detection Engine (Vectorized — Memory-Safe)
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Classify each flight's disruption type using both the BTS-
#          reported delay cause and actual weather conditions at
#          departure. Rewritten using vectorized numpy operations
#          instead of df.apply(axis=1) — the row-by-row .apply() call
#          is what crashed the session last time; np.select() performs
#          the same logic across all 5.9M rows at once, using far less
#          memory and running dramatically faster.
# ═══════════════════════════════════════════════════════════════════════

# ── Weather-Contributed Flag ────────────────────────────────────────────
# True if degraded weather conditions were present at departure time,
# regardless of what BTS listed as the "official" primary cause.
bts_weather['WeatherContributed'] = (
    (bts_weather['IsLowVis'] == 1) |
    (bts_weather['IsHighWind'] == 1) |
    (bts_weather['IsGust'] == 1) |
    (bts_weather['IsFogOrIFR'] == 1)
).astype(int)

# ── Refined Disruption Classification (vectorized) ──────────────────────
# Conditions checked in priority order — first matching condition wins,
# same logic as the row-by-row version, just evaluated column-wise
# across the whole dataset at once instead of row-by-row.
conditions = [
    bts_weather['Cancelled'] == 1,
    bts_weather['Diverted'] == 1,
    bts_weather['ArrDelayMinutes'] < 15,
    bts_weather['WeatherContributed'] == 1,
    bts_weather['PrimaryDelayCause'].notna(),
]

choices = [
    'Cancelled',
    'Diverted',
    'On Time',
    'Weather-Related Delay',
    bts_weather['PrimaryDelayCause'].astype(str) + ' Delay',
]

bts_weather['DisruptionType'] = np.select(conditions, choices, default='Other Delay')

# ── Cascade Risk Flag ────────────────────────────────────────────────────
# Flights with LateAircraftDelay > 0 indicate the delay is a downstream
# effect of a previous flight's lateness (incoming aircraft was late).
bts_weather['IsCascadeRisk'] = (bts_weather['LateAircraftDelay'] > 0).astype(int)

# ── Summary ──────────────────────────────────────────────────────────────
print('Disruption Type breakdown:')
print(bts_weather['DisruptionType'].value_counts().to_string())
print()
print(f'Weather-contributed flights: {bts_weather["WeatherContributed"].sum():,} '
      f'({bts_weather["WeatherContributed"].mean()*100:.2f}%)')
print(f'Cascade-risk flights (late aircraft): {bts_weather["IsCascadeRisk"].sum():,} '
      f'({bts_weather["IsCascadeRisk"].mean()*100:.2f}%)')

Disruption Type breakdown:
DisruptionType
On Time                  4525651
Carrier Delay             432795
Late Aircraft Delay       392168
NAS Delay                 288989
Cancelled                 104301
Weather-Related Delay      99051
Weather Delay              36421
Diverted                   14775
Security Delay              2453
Other Delay                    2

Weather-contributed flights: 319,563 (5.42%)
Cascade-risk flights (late aircraft): 564,889 (9.58%)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Save Disruption-Classified Dataset Locally, Then Upload to Hub
# ═══════════════════════════════════════════════════════════════════════
detected_path = DIR_PROCESSED / 'bts_disruption_detected.parquet'
bts_weather.to_parquet(detected_path, index=False)
print(f'Saved: {detected_path}')

hf_api.upload_file(
    path_or_fileobj=str(detected_path),
    path_in_repo="bts_disruption_detected.parquet",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)
print("✅ bts_disruption_detected.parquet uploaded to Hugging Face Hub")

Saved: /content/airline-disruption/data/processed/bts_disruption_detected.parquet


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...sruption_detected.parquet:  14%|#3        | 16.0MB /  115MB            

✅ bts_disruption_detected.parquet uploaded to Hugging Face Hub


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Train XGBoost Delay Prediction Model
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Predict DepDel15 (departure delay >= 15 min) using only
#          features known BEFORE departure — no post-outcome fields
#          (ArrDelayMinutes, CarrierDelay, LateAircraftDelay, etc.),
#          which would leak the outcome into the prediction.
#
# Train/test split: by TIME (train 2022-2023, test 2024), not random —
#          this mimics real deployment (predicting future disruptions
#          from historical patterns) and avoids inflated accuracy from
#          correlated same-day flights leaking across a random split.
# ═══════════════════════════════════════════════════════════════════════

!pip install -q xgboost scikit-learn

import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report

# ── Select pre-departure-known features only ────────────────────────────
feature_cols = [
    'Airline', 'Origin', 'Month', 'DayOfWeek', 'ScheduledDepHour',
    'Season', 'IsWeekend', 'Distance',
    'TempC', 'WindSpeedKt', 'WindGustKt', 'VisibSM',
    'IsLowVis', 'IsHighWind', 'IsGust', 'IsFogOrIFR', 'FlightCategory',
]
target_col = 'DepDel15'

model_df = bts_weather[feature_cols + [target_col, 'Year']].copy()

# Convert categorical columns to pandas 'category' dtype — XGBoost's
# native categorical support (enable_categorical=True) handles these
# directly without needing memory-heavy one-hot encoding.
cat_cols = ['Airline', 'Origin', 'Season', 'FlightCategory']
for col in cat_cols:
    model_df[col] = model_df[col].astype('category')

# ── Time-based train/test split ──────────────────────────────────────────
train_df = model_df[model_df['Year'] < 2024]
test_df  = model_df[model_df['Year'] == 2024]

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test   = test_df[feature_cols], test_df[target_col]

print(f'Train set: {len(X_train):,} flights (2022-2023)')
print(f'Test set:  {len(X_test):,} flights (2024)')

# ── Train the model ──────────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',        # memory-efficient histogram-based training
    enable_categorical=True,   # native categorical support, no one-hot needed
    eval_metric='auc',
    random_state=42,
)

xgb_model.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
y_pred = xgb_model.predict(X_test)

auc = roc_auc_score(y_test, y_pred_proba)
print(f'\nROC-AUC: {auc:.4f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['On Time', 'Delayed']))

# ── Feature importance ────────────────────────────────────────────────────
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print('\nFeature importance:')
print(importance.to_string(index=False))

Train set: 3,881,146 flights (2022-2023)
Test set:  2,015,460 flights (2024)

ROC-AUC: 0.6728

Classification report:
              precision    recall  f1-score   support

     On Time       0.79      0.99      0.88   1579892
     Delayed       0.51      0.04      0.08    435568

    accuracy                           0.78   2015460
   macro avg       0.65      0.52      0.48   2015460
weighted avg       0.73      0.78      0.71   2015460


Feature importance:
         feature  importance
ScheduledDepHour    0.257479
          Season    0.194746
         Airline    0.114879
         VisibSM    0.091887
          Origin    0.076276
       DayOfWeek    0.057486
        Distance    0.053991
           Month    0.038995
           TempC    0.037188
     WindSpeedKt    0.034888
      WindGustKt    0.026027
  FlightCategory    0.016158
       IsWeekend    0.000000
        IsLowVis    0.000000
      IsHighWind    0.000000
          IsGust    0.000000
      IsFogOrIFR    0.000000


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Retrain XGBoost with Class Imbalance Handling
# ═══════════════════════════════════════════════════════════════════════
# Purpose: The first model had strong overall accuracy (78%) but only
#          4% recall on actual delays — it was essentially predicting
#          "on time" for almost everything, since ~80% of flights ARE
#          on time and unweighted training rewards that shortcut.
#          scale_pos_weight tells XGBoost to weight the minority class
#          (delayed flights) more heavily during training.
#
#          Also dropping the four binary weather flags (IsLowVis,
#          IsHighWind, IsGust, IsFogOrIFR) — they scored zero
#          importance since they're redundant with the continuous
#          VisibSM/WindSpeedKt/WindGustKt features already in the model.
# ═══════════════════════════════════════════════════════════════════════

# Calculate class imbalance ratio for scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f'Class balance — On Time: {neg_count:,}, Delayed: {pos_count:,}')
print(f'scale_pos_weight: {scale_pos_weight:.3f}')

# Drop redundant zero-importance binary flags
feature_cols_v2 = [
    'Airline', 'Origin', 'Month', 'DayOfWeek', 'ScheduledDepHour',
    'Season', 'IsWeekend', 'Distance',
    'TempC', 'WindSpeedKt', 'WindGustKt', 'VisibSM', 'FlightCategory',
]

X_train_v2, X_test_v2 = X_train[feature_cols_v2], X_test[feature_cols_v2]

xgb_model_v2 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',
    enable_categorical=True,
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight,   # ← key fix for class imbalance
    random_state=42,
)

xgb_model_v2.fit(X_train_v2, y_train)

y_pred_proba_v2 = xgb_model_v2.predict_proba(X_test_v2)[:, 1]
y_pred_v2 = xgb_model_v2.predict(X_test_v2)

auc_v2 = roc_auc_score(y_test, y_pred_proba_v2)
print(f'\nROC-AUC: {auc_v2:.4f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred_v2, target_names=['On Time', 'Delayed']))

importance_v2 = pd.DataFrame({
    'feature': feature_cols_v2,
    'importance': xgb_model_v2.feature_importances_
}).sort_values('importance', ascending=False)
print('\nFeature importance:')
print(importance_v2.to_string(index=False))

Class balance — On Time: 3,077,487, Delayed: 803,659
scale_pos_weight: 3.829

ROC-AUC: 0.6724

Classification report:
              precision    recall  f1-score   support

     On Time       0.85      0.67      0.75   1579892
     Delayed       0.33      0.57      0.41    435568

    accuracy                           0.65   2015460
   macro avg       0.59      0.62      0.58   2015460
weighted avg       0.74      0.65      0.68   2015460


Feature importance:
         feature  importance
ScheduledDepHour    0.269493
          Season    0.204914
         Airline    0.109794
         VisibSM    0.093858
          Origin    0.072474
       DayOfWeek    0.054253
        Distance    0.052273
           Month    0.035835
           TempC    0.034549
     WindSpeedKt    0.034530
      WindGustKt    0.025187
  FlightCategory    0.012840
       IsWeekend    0.000000


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Environment Setup
# ═══════════════════════════════════════════════════════════════════════
import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

!pip install -q huggingface_hub

from huggingface_hub import HfApi, login, hf_hub_download
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

HF_REPO_ID = "Dev123Hug456Face/airline-disruption-data"
hf_api = HfApi()

BASE_DIR       = Path('/content/airline-disruption')
DIR_RAW        = BASE_DIR / 'data' / 'raw'
DIR_PROCESSED  = BASE_DIR / 'data' / 'processed'
DIR_WEATHER    = BASE_DIR / 'data' / 'weather'

for d in [DIR_RAW, DIR_PROCESSED, DIR_WEATHER]:
    d.mkdir(parents=True, exist_ok=True)

print('Environment ready.')

Environment ready.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Pull Disruption-Classified Dataset from Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Restore the dataset with DisruptionType, WeatherContributed,
#          and IsCascadeRisk already built in — skips re-running the
#          join and detection engine steps.
# ═══════════════════════════════════════════════════════════════════════

detected_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="bts_disruption_detected.parquet",
    repo_type="dataset",
    local_dir=DIR_PROCESSED,
)

bts_weather = pd.read_parquet(detected_path)

print(f"Downloaded to: {detected_path}")
print(f"Rows: {len(bts_weather):,}")
print(f"Columns: {bts_weather.shape[1]}")
print(f"Memory usage: {bts_weather.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Downloaded to: /content/airline-disruption/data/processed/bts_disruption_detected.parquet
Rows: 5,896,606
Columns: 53
Memory usage: 1.96 GB


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Engineer Congestion and Rolling Delay-Rate Features
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Add two new predictive features, built entirely from data
#          already in hand (no new source data needed):
#
#          1. AirportHourCongestion — how many flights are scheduled
#             to depart the same airport in the same hour. A proxy
#             for traffic volume/runway contention.
#
#          2. RollingDelayRate — the airline's delay rate at that
#             airport over the PRIOR 24 hours. A proxy for network
#             stress/cascading disruption already in progress —
#             computed using only PAST data relative to each flight,
#             so it doesn't leak future information.
# ═══════════════════════════════════════════════════════════════════════

import gc

# ── Feature 1: Airport-hour congestion ──────────────────────────────────
# Count how many flights are scheduled to depart each (Origin, FlightDate,
# ScheduledDepHour) combination — simple groupby + transform, fully
# vectorized, no row-by-row iteration.
bts_weather['AirportHourCongestion'] = bts_weather.groupby(
    ['Origin', 'FlightDate', 'ScheduledDepHour']
)['Origin'].transform('count')

print('AirportHourCongestion computed.')
print(bts_weather['AirportHourCongestion'].describe())

# ── Feature 2: Rolling 24-hour delay rate (per airline + airport) ──────
# For each flight, look at that SAME airline+airport's delay rate over
# the prior 24 hours. Must be sorted by time first, and computed using
# only PAST flights relative to each row (shift(1) + rolling) to avoid
# leaking same-day/future information into the feature.
bts_weather = bts_weather.sort_values(['Airline', 'Origin', 'FlightDate', 'ScheduledDepHour'])

# Daily delay rate per airline+airport+date (aggregated first — much
# cheaper than a row-level rolling window across 5.9M individual flights)
daily_rate = bts_weather.groupby(
    ['Airline', 'Origin', 'FlightDate']
)['DepDel15'].mean().reset_index()
daily_rate = daily_rate.rename(columns={'DepDel15': 'DailyDelayRate'})

# Shift by 1 day so each day's feature reflects the PRIOR day's rate
# (not including that day's own outcomes — avoids leakage)
daily_rate = daily_rate.sort_values(['Airline', 'Origin', 'FlightDate'])
daily_rate['RollingDelayRate'] = daily_rate.groupby(
    ['Airline', 'Origin']
)['DailyDelayRate'].shift(1)

# Merge the lagged daily rate back onto the flight-level data
bts_weather = bts_weather.merge(
    daily_rate[['Airline', 'Origin', 'FlightDate', 'RollingDelayRate']],
    on=['Airline', 'Origin', 'FlightDate'],
    how='left'
)

# Fill missing values (first day of data per airline/airport has no
# prior day) with the overall average delay rate as a reasonable default
overall_avg = bts_weather['DepDel15'].mean()
bts_weather['RollingDelayRate'] = bts_weather['RollingDelayRate'].fillna(overall_avg)

print('\nRollingDelayRate computed.')
print(bts_weather['RollingDelayRate'].describe())

gc.collect()

AirportHourCongestion computed.
count    5.896606e+06
mean     4.043466e+01
std      2.081975e+01
min      1.000000e+00
25%      2.400000e+01
50%      3.500000e+01
75%      5.600000e+01
max      1.080000e+02
Name: AirportHourCongestion, dtype: float64


/tmp/ipykernel_33709/3227172357.py:40: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  daily_rate = bts_weather.groupby(
/tmp/ipykernel_33709/3227172357.py:48: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  daily_rate['RollingDelayRate'] = daily_rate.groupby(



RollingDelayRate computed.
count    5.896606e+06
mean     2.090167e-01
std      1.270156e-01
min      0.000000e+00
25%      1.193548e-01
50%      1.798561e-01
75%      2.680653e-01
max      1.000000e+00
Name: RollingDelayRate, dtype: float64


69

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Save Feature-Enriched Dataset Locally, Then Upload to Hub
# ═══════════════════════════════════════════════════════════════════════
enriched_path = DIR_PROCESSED / 'bts_features_enriched.parquet'
bts_weather.to_parquet(enriched_path, index=False)
print(f'Saved: {enriched_path}')

hf_api.upload_file(
    path_or_fileobj=str(enriched_path),
    path_in_repo="bts_features_enriched.parquet",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)
print("✅ bts_features_enriched.parquet uploaded to Hugging Face Hub")

Saved: /content/airline-disruption/data/processed/bts_features_enriched.parquet


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...features_enriched.parquet:   1%|          |  559kB /  105MB            

✅ bts_features_enriched.parquet uploaded to Hugging Face Hub


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Retrain XGBoost with Congestion + Rolling Delay-Rate Features
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Same setup as the previous model (time-based split, class
#          imbalance handling via scale_pos_weight, pre-departure-only
#          features), now with AirportHourCongestion and
#          RollingDelayRate added — testing whether these two new
#          engineered features improve on the 0.6724 ROC-AUC ceiling
#          from the previous run.
# ═══════════════════════════════════════════════════════════════════════

!pip install -q xgboost scikit-learn

import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report

# ── Feature set: previous features + the two new engineered ones ───────
feature_cols_v3 = [
    'Airline', 'Origin', 'Month', 'DayOfWeek', 'ScheduledDepHour',
    'Season', 'IsWeekend', 'Distance',
    'TempC', 'WindSpeedKt', 'WindGustKt', 'VisibSM', 'FlightCategory',
    'AirportHourCongestion', 'RollingDelayRate',
]
target_col = 'DepDel15'

model_df_v3 = bts_weather[feature_cols_v3 + [target_col, 'Year']].copy()

cat_cols = ['Airline', 'Origin', 'Season', 'FlightCategory']
for col in cat_cols:
    model_df_v3[col] = model_df_v3[col].astype('category')

# ── Time-based train/test split (same as before) ────────────────────────
train_df_v3 = model_df_v3[model_df_v3['Year'] < 2024]
test_df_v3  = model_df_v3[model_df_v3['Year'] == 2024]

X_train_v3, y_train_v3 = train_df_v3[feature_cols_v3], train_df_v3[target_col]
X_test_v3, y_test_v3   = test_df_v3[feature_cols_v3], test_df_v3[target_col]

print(f'Train set: {len(X_train_v3):,} flights (2022-2023)')
print(f'Test set:  {len(X_test_v3):,} flights (2024)')

# Recompute scale_pos_weight for this split
scale_pos_weight_v3 = (y_train_v3 == 0).sum() / (y_train_v3 == 1).sum()

# ── Train ──────────────────────────────────────────────────────────────
xgb_model_v3 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',
    enable_categorical=True,
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight_v3,
    random_state=42,
)

xgb_model_v3.fit(X_train_v3, y_train_v3)

# ── Evaluate ──────────────────────────────────────────────────────────────
y_pred_proba_v3 = xgb_model_v3.predict_proba(X_test_v3)[:, 1]
y_pred_v3 = xgb_model_v3.predict(X_test_v3)

auc_v3 = roc_auc_score(y_test_v3, y_pred_proba_v3)
print(f'\nROC-AUC: {auc_v3:.4f}  (previous: 0.6724)')
print('\nClassification report:')
print(classification_report(y_test_v3, y_pred_v3, target_names=['On Time', 'Delayed']))

importance_v3 = pd.DataFrame({
    'feature': feature_cols_v3,
    'importance': xgb_model_v3.feature_importances_
}).sort_values('importance', ascending=False)
print('\nFeature importance:')
print(importance_v3.to_string(index=False))

Train set: 3,881,146 flights (2022-2023)
Test set:  2,015,460 flights (2024)

ROC-AUC: 0.6904  (previous: 0.6724)

Classification report:
              precision    recall  f1-score   support

     On Time       0.86      0.66      0.75   1579892
     Delayed       0.33      0.62      0.43    435568

    accuracy                           0.65   2015460
   macro avg       0.60      0.64      0.59   2015460
weighted avg       0.75      0.65      0.68   2015460


Feature importance:
              feature  importance
     ScheduledDepHour    0.271476
     RollingDelayRate    0.247637
              VisibSM    0.080830
               Season    0.074127
              Airline    0.056624
            DayOfWeek    0.051564
             Distance    0.046484
               Origin    0.046123
                Month    0.027092
                TempC    0.026075
          WindSpeedKt    0.025666
           WindGustKt    0.022729
AirportHourCongestion    0.013881
       FlightCategory    0.009692
    

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Build Relative Congestion Feature
# ═══════════════════════════════════════════════════════════════════════
# Purpose: The raw AirportHourCongestion feature (flight COUNT per
#          airport-hour) contributed little (0.014 importance) —
#          likely because raw count doesn't distinguish "busy for a
#          small airport" from "busy for a big hub." This version
#          normalizes each airport-hour's count against that SPECIFIC
#          airport's own typical volume at that hour, producing a
#          relative congestion ratio (e.g. 1.5 = 50% busier than usual
#          for this airport at this hour) — a better capacity-pressure
#          signal than raw count.
# ═══════════════════════════════════════════════════════════════════════

# Each airport's average flights-per-hour, by hour of day (its "typical"
# baseline load at that specific hour)
airport_hour_baseline = bts_weather.groupby(
    ['Origin', 'ScheduledDepHour']
)['AirportHourCongestion'].transform('mean')

# Relative congestion = actual / typical for that airport+hour
# (>1 means busier than usual, <1 means quieter than usual)
bts_weather['RelativeCongestion'] = (
    bts_weather['AirportHourCongestion'] / airport_hour_baseline
)

print('RelativeCongestion computed.')
print(bts_weather['RelativeCongestion'].describe())

RelativeCongestion computed.
count    5.896606e+06
mean     1.000000e+00
std      1.749045e-01
min      1.581688e-02
25%      8.994289e-01
50%      1.007497e+00
75%      1.103875e+00
max      4.597817e+00
Name: RelativeCongestion, dtype: float64


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Retrain with RelativeCongestion + Compare XGBoost vs LightGBM
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Test whether RelativeCongestion (capacity-normalized) beats
#          raw AirportHourCongestion, and compare XGBoost against
#          LightGBM on the same feature set — both were on the original
#          project architecture diagram as candidate models.
# ═══════════════════════════════════════════════════════════════════════

!pip install -q xgboost lightgbm scikit-learn

import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, classification_report

# ── Feature set: same as v3, but RelativeCongestion replaces the raw count ──
feature_cols_v4 = [
    'Airline', 'Origin', 'Month', 'DayOfWeek', 'ScheduledDepHour',
    'Season', 'IsWeekend', 'Distance',
    'TempC', 'WindSpeedKt', 'WindGustKt', 'VisibSM', 'FlightCategory',
    'RelativeCongestion', 'RollingDelayRate',
]
target_col = 'DepDel15'

model_df_v4 = bts_weather[feature_cols_v4 + [target_col, 'Year']].copy()

cat_cols = ['Airline', 'Origin', 'Season', 'FlightCategory']
for col in cat_cols:
    model_df_v4[col] = model_df_v4[col].astype('category')

train_df_v4 = model_df_v4[model_df_v4['Year'] < 2024]
test_df_v4  = model_df_v4[model_df_v4['Year'] == 2024]

X_train_v4, y_train_v4 = train_df_v4[feature_cols_v4], train_df_v4[target_col]
X_test_v4, y_test_v4   = test_df_v4[feature_cols_v4], test_df_v4[target_col]

scale_pos_weight_v4 = (y_train_v4 == 0).sum() / (y_train_v4 == 1).sum()

# ── Model 1: XGBoost ──────────────────────────────────────────────────────
xgb_model_v4 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',
    enable_categorical=True,
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight_v4,
    random_state=42,
)
xgb_model_v4.fit(X_train_v4, y_train_v4)
y_pred_proba_xgb = xgb_model_v4.predict_proba(X_test_v4)[:, 1]
auc_xgb = roc_auc_score(y_test_v4, y_pred_proba_xgb)

# ── Model 2: LightGBM ──────────────────────────────────────────────────────
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_v4,
    random_state=42,
    verbose=-1,
)
lgb_model.fit(X_train_v4, y_train_v4)
y_pred_proba_lgb = lgb_model.predict_proba(X_test_v4)[:, 1]
auc_lgb = roc_auc_score(y_test_v4, y_pred_proba_lgb)

# ── Compare ──────────────────────────────────────────────────────────────
print('═' * 50)
print('  MODEL COMPARISON')
print('═' * 50)
print(f'XGBoost  ROC-AUC: {auc_xgb:.4f}  (prior best: 0.6904)')
print(f'LightGBM ROC-AUC: {auc_lgb:.4f}')

print('\n--- XGBoost classification report ---')
print(classification_report(y_test_v4, xgb_model_v4.predict(X_test_v4), target_names=['On Time', 'Delayed']))

print('\n--- LightGBM classification report ---')
print(classification_report(y_test_v4, lgb_model.predict(X_test_v4), target_names=['On Time', 'Delayed']))

print('\n--- XGBoost feature importance ---')
print(pd.DataFrame({'feature': feature_cols_v4, 'importance': xgb_model_v4.feature_importances_})
      .sort_values('importance', ascending=False).to_string(index=False))

══════════════════════════════════════════════════
  MODEL COMPARISON
══════════════════════════════════════════════════
XGBoost  ROC-AUC: 0.6903  (prior best: 0.6904)
LightGBM ROC-AUC: 0.6895

--- XGBoost classification report ---
              precision    recall  f1-score   support

     On Time       0.86      0.66      0.75   1579892
     Delayed       0.33      0.62      0.43    435568

    accuracy                           0.65   2015460
   macro avg       0.60      0.64      0.59   2015460
weighted avg       0.75      0.65      0.68   2015460


--- LightGBM classification report ---
              precision    recall  f1-score   support

     On Time       0.86      0.65      0.74   1579892
     Delayed       0.33      0.62      0.43    435568

    accuracy                           0.65   2015460
   macro avg       0.60      0.64      0.59   2015460
weighted avg       0.75      0.65      0.68   2015460


--- XGBoost feature importance ---
           feature  importance
  Sched

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Configuration + Download BTS Raw Data (PREZIP direct URLs)
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Re-download the 36 raw BTS ZIP files to local disk, so we can
#          re-extract them with Tail_Number included this time (it
#          exists in the raw data but wasn't kept in the original
#          KEEP_COLS list).
# ═══════════════════════════════════════════════════════════════════════

TARGET_AIRPORTS = ['JFK', 'ORD', 'ATL', 'LAX', 'DFW', 'SFO', 'EWR', 'MIA', 'SEA', 'BOS']
START_YEAR, START_MONTH = 2022, 1
END_YEAR, END_MONTH     = 2024, 12

def download_bts_month(year, month):
    """Download one month via the BTS PREZIP direct URL. Returns local path or None."""
    out_path = DIR_RAW / f'BTS_OTP_{year}_{month:02d}.zip'
    if out_path.exists() and out_path.stat().st_size > 10_000:
        print(f'  [cached] {year}-{month:02d}')
        return out_path

    url = (
        f'https://transtats.bts.gov/PREZIP/'
        f'On_Time_Marketing_Carrier_On_Time_Performance_'
        f'Beginning_January_2018_{year}_{month}.zip'
    )

    print(f'  [fetch]  {year}-{month:02d} ...', end=' ', flush=True)
    try:
        r = requests.get(url, timeout=120,
            headers={'User-Agent': 'Mozilla/5.0 (airline-research-project)'})
        r.raise_for_status()
        out_path.write_bytes(r.content)
        kb = out_path.stat().st_size // 1024
        print(f'OK ({kb:,} KB)')
        time.sleep(1)  # polite pause between requests
        return out_path
    except Exception as e:
        print(f'FAILED: {e}')
        if out_path.exists() and out_path.stat().st_size < 1000:
            out_path.unlink()
        return None


print('=== Downloading BTS data (PREZIP direct URLs) ===')
print(f'Range: {START_YEAR}-{START_MONTH:02d} → {END_YEAR}-{END_MONTH:02d}\n')

downloaded, failed = [], []
for year in range(START_YEAR, END_YEAR + 1):
    m_start = START_MONTH if year == START_YEAR else 1
    m_end   = END_MONTH   if year == END_YEAR   else 12
    for month in range(m_start, m_end + 1):
        p = download_bts_month(year, month)
        (downloaded if p else failed).append(f'{year}-{month:02d}')

print(f'\n✅ Downloaded/cached: {len(downloaded)}')
if failed:
    print(f'❌ Failed: {failed}')

=== Downloading BTS data (PREZIP direct URLs) ===
Range: 2022-01 → 2024-12

  [fetch]  2022-01 ... OK (30,031 KB)
  [fetch]  2022-02 ... OK (26,313 KB)
  [fetch]  2022-03 ... OK (30,238 KB)
  [fetch]  2022-04 ... OK (29,917 KB)
  [fetch]  2022-05 ... OK (30,965 KB)
  [fetch]  2022-06 ... OK (31,135 KB)
  [fetch]  2022-07 ... OK (31,794 KB)
  [fetch]  2022-08 ... OK (31,236 KB)
  [fetch]  2022-09 ... OK (29,413 KB)
  [fetch]  2022-10 ... OK (30,050 KB)
  [fetch]  2022-11 ... OK (29,162 KB)
  [fetch]  2022-12 ... OK (29,921 KB)
  [fetch]  2023-01 ... OK (29,875 KB)
  [fetch]  2023-02 ... OK (27,299 KB)
  [fetch]  2023-03 ... OK (31,748 KB)
  [fetch]  2023-04 ... OK (30,772 KB)
  [fetch]  2023-05 ... OK (31,184 KB)
  [fetch]  2023-06 ... OK (31,861 KB)
  [fetch]  2023-07 ... OK (33,073 KB)
  [fetch]  2023-08 ... OK (32,357 KB)
  [fetch]  2023-09 ... OK (30,743 KB)
  [fetch]  2023-10 ... OK (32,251 KB)
  [fetch]  2023-11 ... OK (30,344 KB)
  [fetch]  2023-12 ... OK (31,655 KB)
  [fetch]  2

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Install fastparquet (required for parquet append mode)
# ═══════════════════════════════════════════════════════════════════════
!pip install -q fastparquet
print('fastparquet installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 31.3 MB/s eta 0:00:00
fastparquet installed.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Clean BTS Data — Now Including Tail_Number
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Re-parse all 36 raw ZIPs, this time keeping Tail_Number
#          (individual aircraft ID) — needed to build a true
#          aircraft-chain delay feature. Writes to LOCAL disk first,
#          verifies it's readable, THEN this becomes the working
#          dataset (no Drive involved anywhere in this process).
# ═══════════════════════════════════════════════════════════════════════

KEEP_COLS = [
    'Year', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
    'IATA_Code_Marketing_Airline',
    'Flight_Number_Marketing_Airline',
    'Operating_Airline ',
    'IATA_Code_Operating_Airline',
    'Tail_Number',                          # ← NEW: individual aircraft ID
    'Origin', 'OriginCityName', 'OriginState',
    'Dest',   'DestCityName',   'DestState',
    'CRSDepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15',
    'TaxiOut', 'TaxiIn',
    'CRSArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15',
    'Cancelled', 'CancellationCode', 'Diverted',
    'Distance', 'AirTime',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
]

def process_one_zip(zf_path):
    try:
        with zipfile.ZipFile(zf_path, 'r') as z:
            csv_name = [n for n in z.namelist() if n.endswith('.csv')][0]
            with z.open(csv_name) as f:
                raw_peek = pd.read_csv(f, nrows=0, low_memory=False)
                raw_cols = {c.strip(): c for c in raw_peek.columns}
                use_cols = [raw_cols[k.strip()] for k in KEEP_COLS if k.strip() in raw_cols]

        with zipfile.ZipFile(zf_path, 'r') as z:
            csv_name = [n for n in z.namelist() if n.endswith('.csv')][0]
            with z.open(csv_name) as f:
                df = pd.read_csv(f, usecols=use_cols, low_memory=False)
    except Exception as e:
        print(f'  [error] {zf_path.name}: {e}')
        return None

    df.columns = df.columns.str.strip()
    df = df.rename(columns={
        'IATA_Code_Marketing_Airline':   'Airline',
        'Flight_Number_Marketing_Airline': 'FlightNumber',
        'Operating_Airline':             'OperatingAirline',
        'IATA_Code_Operating_Airline':   'OperatingAirlineCode',
    })

    mask = df['Origin'].isin(TARGET_AIRPORTS) | df['Dest'].isin(TARGET_AIRPORTS)
    df = df[mask].copy()
    return df if len(df) > 0 else None


def clean_chunk(df):
    df['FlightDate'] = pd.to_datetime(df['FlightDate'], errors='coerce')
    df = df.dropna(subset=['FlightDate'])

    num_cols = [
        'DepDelay','DepDelayMinutes','ArrDelay','ArrDelayMinutes',
        'CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay',
        'TaxiOut','TaxiIn','AirTime','Distance',
    ]
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in ['Cancelled','Diverted','DepDel15','ArrDel15']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    if 'CancellationCode' in df.columns:
        df['CancellationCode'] = df['CancellationCode'].str.strip().replace('', np.nan)
        df['CancellationReason'] = df['CancellationCode'].map(
            {'A':'Carrier','B':'Weather','C':'NAS','D':'Security'})

    cause_cols = ['CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay']
    for col in cause_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    df['PrimaryDelayCause'] = df[cause_cols].idxmax(axis=1).where(
        df[cause_cols].max(axis=1) > 0, other=np.nan
    ).map({
        'CarrierDelay':'Carrier','WeatherDelay':'Weather',
        'NASDelay':'NAS','SecurityDelay':'Security','LateAircraftDelay':'Late Aircraft'
    })

    def severity(row):
        if row['Cancelled'] == 1:       return 'Cancelled'
        d = row.get('ArrDelayMinutes', 0) or 0
        if d >= 120: return 'Severe'
        if d >= 45:  return 'Significant'
        if d >= 15:  return 'Minor'
        return 'On Time'
    df['SeverityTier'] = df.apply(severity, axis=1)

    df['ScheduledDepHour'] = pd.to_numeric(
        df['CRSDepTime'].astype(str).str.zfill(4).str[:2], errors='coerce'
    )
    df['Season'] = df['Month'].map({
        12:'Winter',1:'Winter',2:'Winter',
        3:'Spring',4:'Spring',5:'Spring',
        6:'Summer',7:'Summer',8:'Summer',
        9:'Fall',10:'Fall',11:'Fall'
    })
    df['IsWeekend'] = df['DayOfWeek'].isin([6,7]).astype(int)

    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype('float32')

    if 'Year' in df.columns:
        df['Year'] = df['Year'].astype('int16')

    for col in ['Month','DayofMonth','DayOfWeek',
                'DepDel15','ArrDel15','Cancelled','Diverted','IsWeekend']:
        if col in df.columns:
            df[col] = df[col].astype('int8')

    return df.reset_index(drop=True)


# ── Process all ZIPs, write to LOCAL disk first ─────────────────────────
print('Re-processing all ZIPs with Tail_Number included...\n')

zip_files = sorted(DIR_RAW.glob('BTS_OTP_*.zip'))
local_out = DIR_PROCESSED / 'bts_cleaned_with_tail.parquet'

if local_out.exists():
    local_out.unlink()

total_rows = 0
for i, zf in enumerate(zip_files):
    chunk = process_one_zip(zf)
    if chunk is None:
        continue
    chunk = clean_chunk(chunk)
    total_rows += len(chunk)

    if i == 0:
        chunk.to_parquet(local_out, index=False, engine='fastparquet')
    else:
        chunk.to_parquet(local_out, index=False, engine='fastparquet', append=True)

    print(f'  [{i+1:02d}/{len(zip_files)}] {zf.name}: {len(chunk):,} rows  '
          f'(running total: {total_rows:,})')
    del chunk

print(f'\n✅ Done. Total rows: {total_rows:,}')

# ── Verify readability BEFORE trusting this file ────────────────────────
df_tail = pd.read_parquet(local_out)
print(f'Verified — shape: {df_tail.shape}')
print(f'Tail_Number sample: {df_tail["Tail_Number"].dropna().head(3).tolist()}')
print(f'Non-null Tail_Number: {df_tail["Tail_Number"].notna().sum():,} / {len(df_tail):,}')

Re-processing all ZIPs with Tail_Number included...

  [01/36] BTS_OTP_2022_01.zip: 275,486 rows  (running total: 275,486)
  [02/36] BTS_OTP_2022_02.zip: 252,558 rows  (running total: 528,044)
  [03/36] BTS_OTP_2022_03.zip: 284,959 rows  (running total: 813,003)
  [04/36] BTS_OTP_2022_04.zip: 283,691 rows  (running total: 1,096,694)
  [05/36] BTS_OTP_2022_05.zip: 297,709 rows  (running total: 1,394,403)
  [06/36] BTS_OTP_2022_06.zip: 293,584 rows  (running total: 1,687,987)
  [07/36] BTS_OTP_2022_07.zip: 299,820 rows  (running total: 1,987,807)
  [08/36] BTS_OTP_2022_08.zip: 299,291 rows  (running total: 2,287,098)
  [09/36] BTS_OTP_2022_09.zip: 283,206 rows  (running total: 2,570,304)
  [10/36] BTS_OTP_2022_10.zip: 288,520 rows  (running total: 2,858,824)
  [11/36] BTS_OTP_2022_11.zip: 273,728 rows  (running total: 3,132,552)
  [12/36] BTS_OTP_2022_12.zip: 276,648 rows  (running total: 3,409,200)
  [13/36] BTS_OTP_2023_01.zip: 275,102 rows  (running total: 3,684,302)
  [14/36] BTS_OTP

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Upload BTS-with-Tail-Number Dataset to Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
hf_api.upload_file(
    path_or_fileobj=str(local_out),
    path_in_repo="bts_cleaned_with_tail.parquet",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)
print("✅ bts_cleaned_with_tail.parquet uploaded to Hugging Face Hub")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...cleaned_with_tail.parquet:   4%|3         | 14.5MB /  383MB            

✅ bts_cleaned_with_tail.parquet uploaded to Hugging Face Hub


8/14/2026

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Environment Setup
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Set up local working directories and authenticate with
#          Hugging Face Hub. No Google Drive used anywhere — all
#          working data lives on local Colab disk (/content), and
#          finished outputs are pushed to Hugging Face Hub for
#          permanent storage.
# ═══════════════════════════════════════════════════════════════════════

import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

!pip install -q huggingface_hub fastparquet

from huggingface_hub import HfApi, login, hf_hub_download
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

HF_REPO_ID = "Dev123Hug456Face/airline-disruption-data"
hf_api = HfApi()

BASE_DIR       = Path('/content/airline-disruption')
DIR_RAW        = BASE_DIR / 'data' / 'raw'
DIR_PROCESSED  = BASE_DIR / 'data' / 'processed'
DIR_WEATHER    = BASE_DIR / 'data' / 'weather'

for d in [DIR_RAW, DIR_PROCESSED, DIR_WEATHER]:
    d.mkdir(parents=True, exist_ok=True)

TARGET_AIRPORTS = ['JFK', 'ORD', 'ATL', 'LAX', 'DFW', 'SFO', 'EWR', 'MIA', 'SEA', 'BOS']

print('Environment ready.')

Environment ready.


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Pull Data from Hub, Scope to Origin Hubs, Join with Weather
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Restore the tail-number-enriched BTS dataset and weather
#          data from Hugging Face Hub, scope to origin-hub departures,
#          and join — combining three previous steps into one cell.
# ═══════════════════════════════════════════════════════════════════════

# ── Pull BTS data (with Tail_Number) ────────────────────────────────────
tail_path = hf_hub_download(
    repo_id=HF_REPO_ID, filename="bts_cleaned_with_tail.parquet",
    repo_type="dataset", local_dir=DIR_PROCESSED,
)
df_tail = pd.read_parquet(tail_path)
print(f"BTS data: {len(df_tail):,} rows")

# ── Scope to origin-hub departures only ─────────────────────────────────
df_scoped = df_tail[df_tail['Origin'].isin(TARGET_AIRPORTS)].copy()
del df_tail
print(f"Scoped to origin hubs: {len(df_scoped):,} rows")

# ── Pull weather data ────────────────────────────────────────────────────
wx_path = hf_hub_download(
    repo_id=HF_REPO_ID, filename="metar_clean.parquet",
    repo_type="dataset", local_dir=DIR_WEATHER,
)
wx = pd.read_parquet(wx_path)
print(f"Weather data: {len(wx):,} rows")

# ── Join ──────────────────────────────────────────────────────────────
df_scoped['JoinDate'] = df_scoped['FlightDate'].dt.date
bts_weather = df_scoped.merge(
    wx, left_on=['Origin', 'JoinDate', 'ScheduledDepHour'],
    right_on=['IATA', 'Date', 'DepHour'], how='left'
)
bts_weather = bts_weather.drop(columns=['JoinDate', 'IATA', 'Date', 'DepHour'])
del df_scoped

matched = bts_weather['FlightCategory'].notna().sum()
print(f'\nJoined: {len(bts_weather):,} rows, {matched/len(bts_weather)*100:.2f}% weather match')

BTS data: 10,504,936 rows
Scoped to origin hubs: 5,896,606 rows
Weather data: 263,034 rows

Joined: 5,896,606 rows, 100.00% weather match


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Reduce Memory Footprint Before Feature Engineering
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Convert repetitive string/object columns to 'category' dtype
#          before doing any further feature engineering. This is the
#          step we missed in the previous rebuild attempt — going
#          straight from join into feature engineering (with the added
#          weight of the high-cardinality Tail_Number column) is what
#          caused the RAM crash.
# ═══════════════════════════════════════════════════════════════════════

import gc

category_cols = [
    'Airline', 'OperatingAirline', 'OperatingAirlineCode',
    'Origin', 'OriginCityName', 'OriginState',
    'Dest', 'DestCityName', 'DestState',
    'CancellationReason', 'PrimaryDelayCause', 'SeverityTier', 'Season',
    'FlightCategory',
]

for col in category_cols:
    if col in bts_weather.columns:
        bts_weather[col] = bts_weather[col].astype('category')

# NOTE: Tail_Number is NOT converted to category — it has very high
# cardinality (thousands of unique aircraft), so category dtype would
# add overhead rather than save memory. It stays as a string column,
# and we'll only use it briefly for grouping/sorting in the next step.

gc.collect()

print('Memory optimization complete.')
print(f'bts_weather memory usage: {bts_weather.memory_usage(deep=True).sum() / 1e9:.2f} GB')
!free -h

Memory optimization complete.
bts_weather memory usage: 1.54 GB
               total        used        free      shared  buff/cache   available
Mem:            12Gi       5.4Gi       5.4Gi       2.0Mi       1.9Gi       7.0Gi
Swap:             0B          0B          0B


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Disruption Detection Engine (Vectorized)
# ═══════════════════════════════════════════════════════════════════════
bts_weather['WeatherContributed'] = (
    (bts_weather['IsLowVis'] == 1) |
    (bts_weather['IsHighWind'] == 1) |
    (bts_weather['IsGust'] == 1) |
    (bts_weather['IsFogOrIFR'] == 1)
).astype(int)

conditions = [
    bts_weather['Cancelled'] == 1,
    bts_weather['Diverted'] == 1,
    bts_weather['ArrDelayMinutes'] < 15,
    bts_weather['WeatherContributed'] == 1,
    bts_weather['PrimaryDelayCause'].notna(),
]

choices = [
    'Cancelled',
    'Diverted',
    'On Time',
    'Weather-Related Delay',
    bts_weather['PrimaryDelayCause'].astype(str) + ' Delay',
]

bts_weather['DisruptionType'] = np.select(conditions, choices, default='Other Delay')
bts_weather['IsCascadeRisk'] = (bts_weather['LateAircraftDelay'] > 0).astype(int)

print('Disruption Type breakdown:')
print(bts_weather['DisruptionType'].value_counts().to_string())

Disruption Type breakdown:
DisruptionType
On Time                  4525651
Carrier Delay             432795
Late Aircraft Delay       392168
NAS Delay                 288989
Cancelled                 104301
Weather-Related Delay      99051
Weather Delay              36421
Diverted                   14775
Security Delay              2453
Other Delay                    2


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Congestion and Rolling Delay-Rate Features
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Rebuild the two engineered features validated earlier —
#          AirportHourCongestion (flight count per airport-hour) and
#          RollingDelayRate (prior-day airline+airport delay rate,
#          lagged to avoid leakage).
# ═══════════════════════════════════════════════════════════════════════

# ── Feature 1: Airport-hour congestion ──────────────────────────────────
bts_weather['AirportHourCongestion'] = bts_weather.groupby(
    ['Origin', 'FlightDate', 'ScheduledDepHour'], observed=True
)['Origin'].transform('count')

# ── Feature 2: Rolling 24-hour delay rate (per airline + airport) ──────
bts_weather = bts_weather.sort_values(['Airline', 'Origin', 'FlightDate', 'ScheduledDepHour'])

daily_rate = bts_weather.groupby(
    ['Airline', 'Origin', 'FlightDate'], observed=True
)['DepDel15'].mean().reset_index()
daily_rate = daily_rate.rename(columns={'DepDel15': 'DailyDelayRate'})

daily_rate = daily_rate.sort_values(['Airline', 'Origin', 'FlightDate'])
daily_rate['RollingDelayRate'] = daily_rate.groupby(
    ['Airline', 'Origin'], observed=True
)['DailyDelayRate'].shift(1)

bts_weather = bts_weather.merge(
    daily_rate[['Airline', 'Origin', 'FlightDate', 'RollingDelayRate']],
    on=['Airline', 'Origin', 'FlightDate'],
    how='left'
)

overall_avg = bts_weather['DepDel15'].mean()
bts_weather['RollingDelayRate'] = bts_weather['RollingDelayRate'].fillna(overall_avg)

gc.collect()

print('AirportHourCongestion — mean:', bts_weather['AirportHourCongestion'].mean().round(2),
      ' min:', bts_weather['AirportHourCongestion'].min(),
      ' max:', bts_weather['AirportHourCongestion'].max())
print('RollingDelayRate — mean:', bts_weather['RollingDelayRate'].mean().round(4),
      ' min:', bts_weather['RollingDelayRate'].min(),
      ' max:', bts_weather['RollingDelayRate'].max())

AirportHourCongestion — mean: 40.43  min: 1  max: 108
RollingDelayRate — mean: 0.209  min: 0.0  max: 1.0


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Build Aircraft-Chain Delay Feature (Tail_Number)
# ═══════════════════════════════════════════════════════════════════════
# Purpose: For each flight, determine whether that SPECIFIC aircraft's
#          previous leg (by Tail_Number) arrived late. This is a
#          genuinely different signal from RollingDelayRate (which is
#          airline+airport level) — it tracks the individual plane's
#          own rotation through the day.
#
# Important caveat: our dataset is scoped to flights touching the 10
# hub airports, so an aircraft's "previous flight" in this dataset may
# not be its true previous flight in the full national network (if the
# prior leg didn't touch a hub, it's not in our data). This is a known
# limitation — the feature is a defensible on the departures we *did*
# capture at hub airports, but shift(1) will sometimes point to a leg
# that is not physically the immediately prior one.
# ═══════════════════════════════════════════════════════════════════════

# Build a proper chronological ordering key from date + scheduled dep time
bts_weather['DepDateTime'] = pd.to_datetime(
    bts_weather['FlightDate'].astype(str) + ' ' +
    bts_weather['CRSDepTime'].astype(str).str.zfill(4).str[:2] + ':' +
    bts_weather['CRSDepTime'].astype(str).str.zfill(4).str[2:],
    errors='coerce'
)

# Sort by aircraft + chronological order — required before using shift()
bts_weather = bts_weather.sort_values(['Tail_Number', 'DepDateTime'])

# For each aircraft, get the PREVIOUS flight's arrival delay (in minutes)
# and whether it was delayed — shift(1) looks at the row immediately
# before in this sorted order, which for a given Tail_Number is its
# prior leg within our dataset.
bts_weather['PrevLegArrDelay'] = bts_weather.groupby(
    'Tail_Number', observed=True
)['ArrDelayMinutes'].shift(1)

bts_weather['PrevLegWasDelayed'] = bts_weather.groupby(
    'Tail_Number', observed=True
)['ArrDel15'].shift(1)

# Missing values occur for: (a) an aircraft's FIRST flight in our
# dataset (no prior leg to reference), or (b) missing Tail_Number.
# Fill with 0 (treat as "not known to be delayed") — a reasonable,
# conservative default.
bts_weather['PrevLegArrDelay'] = bts_weather['PrevLegArrDelay'].fillna(0)
bts_weather['PrevLegWasDelayed'] = bts_weather['PrevLegWasDelayed'].fillna(0).astype(int)

gc.collect()

print(f'PrevLegWasDelayed — rate: {bts_weather["PrevLegWasDelayed"].mean()*100:.2f}%')
print(f'PrevLegArrDelay — mean: {bts_weather["PrevLegArrDelay"].mean():.2f} min, '
      f'max: {bts_weather["PrevLegArrDelay"].max():.0f} min')
print(f'\nRows with a known previous leg: '
      f'{(bts_weather["PrevLegArrDelay"] != 0).sum() + (bts_weather["PrevLegWasDelayed"] == 0).sum() - len(bts_weather) + (bts_weather["PrevLegArrDelay"] == 0).sum():,}')

PrevLegWasDelayed — rate: 21.21%
PrevLegArrDelay — mean: 15.54 min, max: 5986 min

Rows with a known previous leg: 4,646,229


In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Save Final Feature Set Locally, Then Upload to Hub
# ═══════════════════════════════════════════════════════════════════════
final_path = DIR_PROCESSED / 'bts_features_with_tail.parquet'
bts_weather.to_parquet(final_path, index=False)
print(f'Saved: {final_path}')

hf_api.upload_file(
    path_or_fileobj=str(final_path),
    path_in_repo="bts_features_with_tail.parquet",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)
print("✅ Uploaded to Hugging Face Hub")

Saved: /content/airline-disruption/data/processed/bts_features_with_tail.parquet


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...eatures_with_tail.parquet:   0%|          |  526kB /  227MB            

✅ Uploaded to Hugging Face Hub


In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Retrain XGBoost with Aircraft-Chain Delay Features
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Same setup as before (time-based split, class imbalance
#          handling), now adding PrevLegWasDelayed and PrevLegArrDelay
#          — testing whether the aircraft-chain signal beats the
#          0.6903 ROC-AUC ceiling from the airline/airport-level
#          RollingDelayRate alone.
# ═══════════════════════════════════════════════════════════════════════

!pip install -q xgboost scikit-learn

import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report

# ── Feature set: previous best features + new aircraft-chain features ──
feature_cols_v5 = [
    'Airline', 'Origin', 'Month', 'DayOfWeek', 'ScheduledDepHour',
    'Season', 'IsWeekend', 'Distance',
    'TempC', 'WindSpeedKt', 'WindGustKt', 'VisibSM', 'FlightCategory',
    'AirportHourCongestion', 'RollingDelayRate',
    'PrevLegWasDelayed', 'PrevLegArrDelay',   # ← NEW
]
target_col = 'DepDel15'

model_df_v5 = bts_weather[feature_cols_v5 + [target_col, 'Year']].copy()

cat_cols = ['Airline', 'Origin', 'Season', 'FlightCategory']
for col in cat_cols:
    model_df_v5[col] = model_df_v5[col].astype('category')

# ── Time-based train/test split ──────────────────────────────────────────
train_df_v5 = model_df_v5[model_df_v5['Year'] < 2024]
test_df_v5  = model_df_v5[model_df_v5['Year'] == 2024]

X_train_v5, y_train_v5 = train_df_v5[feature_cols_v5], train_df_v5[target_col]
X_test_v5, y_test_v5   = test_df_v5[feature_cols_v5], test_df_v5[target_col]

print(f'Train set: {len(X_train_v5):,} flights (2022-2023)')
print(f'Test set:  {len(X_test_v5):,} flights (2024)')

scale_pos_weight_v5 = (y_train_v5 == 0).sum() / (y_train_v5 == 1).sum()

# ── Train ──────────────────────────────────────────────────────────────
xgb_model_v5 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',
    enable_categorical=True,
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight_v5,
    random_state=42,
)
xgb_model_v5.fit(X_train_v5, y_train_v5)

# ── Evaluate ──────────────────────────────────────────────────────────────
y_pred_proba_v5 = xgb_model_v5.predict_proba(X_test_v5)[:, 1]
y_pred_v5 = xgb_model_v5.predict(X_test_v5)

auc_v5 = roc_auc_score(y_test_v5, y_pred_proba_v5)
print(f'\nROC-AUC: {auc_v5:.4f}  (previous best: 0.6903)')
print('\nClassification report:')
print(classification_report(y_test_v5, y_pred_v5, target_names=['On Time', 'Delayed']))

importance_v5 = pd.DataFrame({
    'feature': feature_cols_v5,
    'importance': xgb_model_v5.feature_importances_
}).sort_values('importance', ascending=False)
print('\nFeature importance:')
print(importance_v5.to_string(index=False))

Train set: 3,881,146 flights (2022-2023)
Test set:  2,015,460 flights (2024)

ROC-AUC: 0.7222  (previous best: 0.6903)

Classification report:
              precision    recall  f1-score   support

     On Time       0.87      0.72      0.78   1579892
     Delayed       0.37      0.61      0.46    435568

    accuracy                           0.69   2015460
   macro avg       0.62      0.66      0.62   2015460
weighted avg       0.76      0.69      0.71   2015460


Feature importance:
              feature  importance
      PrevLegArrDelay    0.380989
     ScheduledDepHour    0.176415
     RollingDelayRate    0.125303
               Season    0.046610
              VisibSM    0.041848
              Airline    0.037026
            DayOfWeek    0.033775
             Distance    0.030449
               Origin    0.029628
                TempC    0.020779
    PrevLegWasDelayed    0.016665
                Month    0.015627
           WindGustKt    0.014766
          WindSpeedKt    0.014762

In [9]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Save Final Model Metrics (v5 — with Tail_Number features)
# ═══════════════════════════════════════════════════════════════════════
import json

final_metrics = {
    'model': 'XGBoost',
    'roc_auc': round(auc_v5, 4),
    'train_period': '2022-2023',
    'test_period': '2024',
    'train_size': len(X_train_v5),
    'test_size': len(X_test_v5),
    'features_used': feature_cols_v5,
    'feature_importance': dict(zip(
        feature_cols_v5,
        [round(float(x), 4) for x in xgb_model_v5.feature_importances_]
    )),
    'notes': (
        'Final model. Adding Tail_Number-based aircraft-chain features '
        '(PrevLegArrDelay, PrevLegWasDelayed) improved ROC-AUC from '
        '0.6903 to 0.7222 -- the largest single improvement across all '
        'iterations. PrevLegArrDelay (continuous prior-leg delay in '
        'minutes) was the single most important feature (0.381), '
        'far exceeding PrevLegWasDelayed (binary flag, 0.017) -- '
        'delay MAGNITUDE carries far more signal than a simple '
        'threshold flag. Known limitation: dataset is scoped to '
        'flights touching the 10 hub airports, so an aircraft\'s '
        '"previous leg" in this data is not always its true '
        'immediately-prior flight in the full national network.'
    )
}

metrics_path = DIR_PROCESSED / 'model_metrics_final.json'
with open(metrics_path, 'w') as f:
    json.dump(final_metrics, f, indent=2)

hf_api.upload_file(
    path_or_fileobj=str(metrics_path),
    path_in_repo="model_metrics_final.json",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)

print(f'Saved and uploaded: {metrics_path}')
print(json.dumps(final_metrics, indent=2))

Saved and uploaded: /content/airline-disruption/data/processed/model_metrics_final.json
{
  "model": "XGBoost",
  "roc_auc": 0.7222,
  "train_period": "2022-2023",
  "test_period": "2024",
  "train_size": 3881146,
  "test_size": 2015460,
  "features_used": [
    "Airline",
    "Origin",
    "Month",
    "DayOfWeek",
    "ScheduledDepHour",
    "Season",
    "IsWeekend",
    "Distance",
    "TempC",
    "WindSpeedKt",
    "WindGustKt",
    "VisibSM",
    "FlightCategory",
    "AirportHourCongestion",
    "RollingDelayRate",
    "PrevLegWasDelayed",
    "PrevLegArrDelay"
  ],
  "feature_importance": {
    "Airline": 0.037,
    "Origin": 0.0296,
    "Month": 0.0156,
    "DayOfWeek": 0.0338,
    "ScheduledDepHour": 0.1764,
    "Season": 0.0466,
    "IsWeekend": 0.0,
    "Distance": 0.0304,
    "TempC": 0.0208,
    "WindSpeedKt": 0.0148,
    "WindGustKt": 0.0148,
    "VisibSM": 0.0418,
    "FlightCategory": 0.0067,
    "AirportHourCongestion": 0.0086,
    "RollingDelayRate": 0.1253,
    "Pr